# NUEVA PROPUESTA 1/2 — Semana 9: entrenar backbone, CBAM y detector 2.5D

Este notebook es la nueva propuesta de Semana 9. Entrena únicamente los componentes exigidos en esa semana:

```text
[Z-1, Z, Z+1] → FundidoraBackbone → CBAM → cabeza de detección
```

Su checkpoint es la entrada obligatoria del notebook `NUEVA_PROPUESTA_02_S10_multitarea.ipynb`. Esta versión parte de pesos aleatorios y usa rotación física en milímetros desde la primera época. Semana 10 no repite este entrenamiento: carga sus pesos y agrega las cabezas nuevas.

In [ ]:
import os
import sys
import json
import time
import subprocess
from pathlib import Path

def encontrar_raiz(inicio: Path) -> Path:
    for candidato in (inicio, *inicio.parents):
        if (candidato / "src" / "train_week9_detection.py").exists() and (candidato / "README.md").exists():
            return candidato
    raise FileNotFoundError("No encontré la raíz del repositorio.")

ROOT = encontrar_raiz(Path.cwd().resolve())
os.chdir(ROOT)
os.environ["PYTHONUTF8"] = "1"
print("Raíz:", ROOT)
print("Python:", sys.version.split()[0], "| ejecutable:", sys.executable)

## 1. Configuración

Ejecuta primero el modo rápido en ambos notebooks. Cuando funcione, cambia `MODO_RAPIDO=False` también en ambos para la corrida completa.

In [ ]:
MODO_RAPIDO = False
EXIGIR_GPU = True
PREPARAR_CACHE = True
RECONSTRUIR_CACHE = False
EJECUTAR_ENTRENAMIENTO = True

CONTEXT = 1
SIZE = 384
SEED = 42
SALIDA_DIR = ROOT / "outputs" / "week9_25d"
SALIDA_DIR.mkdir(parents=True, exist_ok=True)

if MODO_RAPIDO:
    NOMBRE = "smoke_week9_25d_rotacion_fisica_desde_cero_w8_e2_s42"
    CACHE = ROOT / "data" / "cache" / "week9_detection_25d_fisico_smoke_384.npz"
    EPOCHS, BASE_CHANNELS, BATCH_SIZE = 2, 8, 4
    TRAIN_CASES, VAL_CASES = 4, 2
    SLICES_PER_CASE, NEGATIVES_PER_CASE = 2, 1
else:
    NOMBRE = "week9_25d_rotacion_fisica_desde_cero_w32_e35_s42"
    CACHE = ROOT / "data" / "cache" / "week9_detection_25d_fisico_384.npz"
    EPOCHS, BASE_CHANNELS, BATCH_SIZE = 35, 32, 4
    TRAIN_CASES, VAL_CASES = 70, 15
    SLICES_PER_CASE, NEGATIVES_PER_CASE = 12, 2

CHECKPOINT = SALIDA_DIR / f"{NOMBRE}.pth"
HISTORIAL = CHECKPOINT.with_suffix(".json")
LOG = SALIDA_DIR / f"{NOMBRE}_train.log"
print("Modo:", "RÁPIDO" if MODO_RAPIDO else "COMPLETO")
print("Caché:", CACHE)
print("Checkpoint que consumirá Semana 10:", CHECKPOINT)

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
elif EXIGIR_GPU:
    raise RuntimeError("No hay GPU CUDA. Selecciona el kernel del entorno .venv con PyTorch CUDA.")

In [ ]:
def ejecutar_visible(comando, titulo, log_path=None):
    print(f"\n=== {titulo} ===")
    print(" ".join(map(str, comando)))
    inicio = time.time()
    with subprocess.Popen(comando, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as proceso:
        lineas = []
        for linea in proceso.stdout:
            print(linea, end="")
            lineas.append(linea)
        codigo = proceso.wait()
    if log_path is not None:
        log_path.write_text("".join(lineas), encoding="utf-8")
    print(f"=== finalizó con código {codigo} en {(time.time()-inicio)/60:.1f} min ===")
    if codigo != 0:
        raise RuntimeError(f"El comando terminó con código {codigo}.")

## 2. Preparar el caché 2.5D

Los tres canales se construyen antes del entrenamiento y usan la misma orientación por metadatos de Semana 10. El caché guarda `spacing_yx` por muestra; durante entrenamiento los tres canales y la máscara giran juntos en milímetros. El target siempre corresponde al corte central `Z`.

In [ ]:
comando_cache = [
    sys.executable, "src/prepare_week9_detection_data.py",
    "--output", str(CACHE), "--size", str(SIZE),
    "--context", str(CONTEXT), "--canonical",
    "--slices-per-case", str(SLICES_PER_CASE),
    "--negatives-per-case", str(NEGATIVES_PER_CASE),
    "--max-train-cases", str(TRAIN_CASES), "--max-val-cases", str(VAL_CASES),
    "--extent", "0.0", "1.0",
]
if RECONSTRUIR_CACHE and CACHE.exists():
    CACHE.unlink()
if PREPARAR_CACHE and not CACHE.exists():
    ejecutar_visible(comando_cache, "Preparación caché Semana 9 2.5D")
elif CACHE.exists():
    print("Se reutiliza el caché:", CACHE)
else:
    raise FileNotFoundError("No existe el caché y PREPARAR_CACHE=False.")

In [ ]:
import numpy as np
with np.load(CACHE) as datos:
    print("images:", datos["images"].shape)
    print("labels:", datos["labels"].shape)
    print("context:", int(datos["context"]))
    print("canonical:", bool(datos["canonical"]))
    print("spacing_yx:", datos["spacing_yx"].shape, "rango", datos["spacing_yx"].min(axis=0), datos["spacing_yx"].max(axis=0))
    assert datos["images"].shape[1] == 3
    assert int(datos["context"]) == 1
    assert bool(datos["canonical"])
    assert datos["spacing_yx"].shape == (len(datos["images"]), 2)
print("OK: entrada Semana 9 = [Z-1, Z, Z+1].")

## 3. Entrenar backbone, CBAM y detector

In [ ]:
comando_entrenamiento = [
    sys.executable, "src/train_week9_detection.py",
    "--cache", str(CACHE), "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE), "--base-channels", str(BASE_CHANNELS),
    "--cbam-stages", "1", "--seed", str(SEED),
    "--aug-geo", "1.0", "--aug-rot-deg", "35", "--aug-mm",
    "--disable-cudnn", "--output", str(CHECKPOINT),
]
if EJECUTAR_ENTRENAMIENTO:
    ejecutar_visible(comando_entrenamiento, "Entrenamiento Semana 9 2.5D", LOG)
elif CHECKPOINT.exists():
    print("Se reutiliza:", CHECKPOINT)
else:
    raise FileNotFoundError("No existe el checkpoint y EJECUTAR_ENTRENAMIENTO=False.")

## 4. Verificar el artefacto que recibe Semana 10

In [ ]:
contenido = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
assert int(contenido["in_channels"]) == 3
assert int(contenido["context"]) == 1
assert bool(contenido["canonical"])
assert int(contenido["base_channels"]) == BASE_CHANNELS
assert bool(contenido["aug_mm"]) and float(contenido["aug_rot_deg"]) == 35.0
estado = contenido["model"]
backbone = [k for k in estado if k.startswith("backbone.")]
detector = [k for k in estado if k.startswith("detection_head.")]
no_finitos = sum(int((~torch.isfinite(v)).sum()) for v in estado.values()
                 if torch.is_tensor(v) and v.is_floating_point())
print("Mejor época:", contenido["best_epoch"])
print("F1 validación:", contenido["best_val_f1"])
print("Tensores backbone/CBAM:", len(backbone))
print("Tensores detector:", len(detector))
print("Valores no finitos:", no_finitos)
assert backbone and detector and no_finitos == 0
print("\nLISTO PARA SEMANA 10:", CHECKPOINT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
hist = json.loads(HISTORIAL.read_text(encoding="utf-8"))
df = pd.DataFrame(hist["historial"])
display(df.tail())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df.epoch, df.train_loss); axes[0].set(title="Pérdida detección", xlabel="Época")
axes[1].plot(df.epoch, df.f1, label="F1"); axes[1].plot(df.epoch, df.iou_medio, label="IoU")
axes[1].set(title="Validación Semana 9", xlabel="Época"); axes[1].legend()
plt.tight_layout(); plt.show()

## Salida contractual

No cierres ni muevas el checkpoint. El notebook siguiente usa exactamente la misma ruta según el valor de `MODO_RAPIDO`.